**2D**
1) Define the domain $\Omega$
2) Generate triangulations T_h
3) Construct finite element space V_h
4) Get local stiffness matrix A_k
5) Get local mass matrix M_k
6) Assemble local stiffness matrices into global stiffness matrix A
7) Assemble local mass matrices into a global mass matrix M
8) Impose Dirichlet Boundary Condition
9) Solve $A * u = \lambda * M * u$
10) Extract eigenvalues
11) compute the eigenvectors
12) Compare with exact solution


**2D FEM Implementation notes**
1. Generated mesh using Delaunay triangulation
2. Computed local element matrices
3. Assembled global stiffness and mass matrices
4. Identified boundaru and interior nodes
5. Imposed homogeneous Dirichlet boundary conditions by removing rows and columns associated with boundary nodes
6. Solved reduced generalised eigenvalue problem

*For future*
- global matrices are sparse (majority of elements are 0) and symmetric
- boundary conditions reduce the dimention of the system
- enumerate shows which node number corresponds to the coordinates
- with 1 interior node we have 1 degree of freedom, the eigenvalue was 48, comparing to the exact first Dirichlet eigenvalue on the unit square is $\lambda_1 = 2 * \pi^2 \approx 19.739$
- the exact eigenvalue for the unit square with Dirichlet BC is $\lambda_m,n = \pi^2 (m^2 + n^2)$
- with a finer mesh, where there are 5 points on the axis the eigenvalue was 22.506 which is much closer to the real one 19.739
- for generalised FEM eigenproblem the common normalisation is $u^T M u = 1$
- shape is more important when printing the eigenvector (it is 0 on the boundary and maximum in the centre), which is the discrete approximation of the first eigenfunction of the unit square $u(x, y) = sin(\pi x) * sin(\pi y)$. If we take the finer mesh the numbers will be the coefficients of the $u_h = \sum_i u_i \phi_i (x)$. 

**The vector tells us how much of each basis function is present in the final FEM approximation**.
*The numbers in the eigenvector are coefficients at the interior nodes, not values at every point of the domain.*

In [ ]:
# 2D let the domain be uniform and defined on [0, 1]x[0, 1]
import numpy as np
from scipy.linalg import eigh
from scipy.spatial import Delaunay
import pandas as pd
import math
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-poster")

In [ ]:
# generate mesh
def generate_mesh(n_points):
       # define the axis intervals
       x = np.linspace(0, 1, n_points)
       y = np.linspace(0, 1, n_points)

       # coordinate generation via meshgrid (takes 1D arrays and duplicates them to build 2D grids)
       X, Y = np.meshgrid(x, y)

       # c_ matches the first X with the first Y, second X with second Y etc
       domain = np.c_[X.ravel(), Y.ravel()]
       # ravel() takes 2D matrix structure and reads it row by row into a long single list of coordinates
       # Delaunay cannot read 2D grid, thats why we flatten X and Y, so they can be paired together

       # create triangles on the domain
       tri = Delaunay(domain)

       return domain, tri

# method for putting values on the main diagonal
def put_value_in_special_index(matrix, grad_phi_i, grad_phi_j, row, column):
       # find the dot product of 2 gradients and put them in the matrix
       matrix[row, column] = np.dot(grad_phi_i, grad_phi_j)
       
       return matrix

# stiffness matrix
def stiffness_matrix_A(grad_phi):
       # create a matrix (interior_nodes x interior_nodes) of zeros
       A_lower_tri = np.zeros((3, 3), dtype=float)
       # off diagonal values
       # first make lower triangular matrix
       # A_2_1
       put_value_in_special_index(A_lower_tri, grad_phi[1], grad_phi[0], 1, 0) 
       # A_3_1
       put_value_in_special_index(A_lower_tri, grad_phi[2], grad_phi[0], 2, 0)
       # A_3_2
       put_value_in_special_index(A_lower_tri, grad_phi[2], grad_phi[1], 2, 1)

       A = create_symmetric_matrix(A_lower_tri)
       # diagonal values
       put_value_in_special_index(A, grad_phi[0], grad_phi[0], 0, 0)
       put_value_in_special_index(A, grad_phi[1], grad_phi[1], 1, 1)
       put_value_in_special_index(A, grad_phi[2], grad_phi[2], 2, 2)
       return A

def create_symmetric_matrix(lower_tri_matrix):
       # Zero teh diagonal so it isnt double counted, when add
       lower_tri_no_diag = lower_tri_matrix.copy()
       np.fill_diagonal(lower_tri_no_diag, 0)
       
       # create symmetric matrix by adding the lower triangular matrix to its transpose
       sym_matrix = lower_tri_no_diag + lower_tri_no_diag.T

       return sym_matrix

# mass matrix
def M_loc():
       
       return np.array(
              [
                     [2, 1, 1],
                     [1, 2, 1],
                     [1, 1, 2]
              ]
       )


# find the stiffness and mass matrices for individual triangle
def triangle_solver(coords_of_triangle):

       # find the area of the triangle
       col = np.array([1, 1, 1])
       # create 3x3 matrix to find the area
       coords_matrix = np.hstack((coords_of_triangle, np.atleast_2d(col).T))

       # area of a triangle
       T_k = 0.5 * abs(np.linalg.det(coords_matrix))

       x = []
       y = []
       # for coordinate in all of the coordinates of the nodes of this triangle
       for coord in coords_of_triangle:
              x.append(float(coord[0]))
              y.append(float(coord[1]))

       c = []   
       c.append(x[2] - x[1])
       c.append(x[0] - x[2])
       c.append(x[1] - x[0])

       b = []
       b.append(y[1] - y[2])
       b.append(y[2] - y[0])
       b.append(y[0] - y[1])

       # find the gradients of the basis functions
       grad_phi = np.array([
              (1 / (2 * T_k)) * np.array([b_i, c_i]) for b_i, c_i in zip(b, c)
       ])


       # get local stiffness matrix
       A_local = T_k * stiffness_matrix_A(grad_phi)
       
       # get local stiffness matrix
       M_local = (T_k / 12) * M_loc()
       
       return A_local, M_local

# put the triangle in the global matrix
def put_local_to_global(global_matrix, local_matrix, coord):

       n_local = local_matrix.shape[0]

       for a in range(n_local):
              for b in range(n_local):
                     global_matrix[coord[a], coord[b]] += local_matrix[a, b]
       

       return global_matrix

# get boundary and interior nodes
def get_boundary_and_interior_nodes(domain):
       boundary_nodes = []
       interior_nodes = []

       for i, (x, y) in enumerate(domain):
              # check if any of the coordinates are on the boundary
              if x == 0 or x == 1 or y == 0 or y == 1:
                     boundary_nodes.append(i)
              else:
                     interior_nodes.append(i)

       return boundary_nodes, interior_nodes

# get global matrices
def get_global_matrices(tri_coord_sort, domain, n_nodes):

       A_global = np.zeros((n_nodes, n_nodes), dtype=float)
       M_global = np.zeros((n_nodes, n_nodes), dtype=float)

       # for every trinagle in the mesh
       for triangle in tri_coord_sort:
              coords = domain[triangle]

              A_local, M_local = triangle_solver(coords)
              global_coords = triangle.tolist()

              put_local_to_global(A_global, A_local, global_coords)
              put_local_to_global(M_global, M_local, global_coords)

       return A_global, M_global

def apply_dirichlet(A_global, M_global, interior_nodes):
       # reducing matrices based on boundary condition, that u = 0 on the boundary
       A_reduced = A_global[np.ix_(interior_nodes, interior_nodes)]
       M_reduced = M_global[np.ix_(interior_nodes, interior_nodes)]

       return A_reduced, M_reduced

def first_eigval_error(comp_eigval):
       real_eigval = 2 * (np.pi)**2
       error = abs(real_eigval - comp_eigval)
       return error

def second_third_eigval_error(comp_eigval_second, comp_eigval_third):
       real_eigval = 5 * (np.pi)**2
       error_second = abs(real_eigval - comp_eigval_second)
       error_third = abs(real_eigval - comp_eigval_third)
       error = (error_second + error_third) / 2
       return error

def get_table(headers, indexes, table):
       
       df = pd.DataFrame(table, columns = headers, index = indexes)
       return df


In [ ]:
# visualisations
# visualise the mesh

def visualise_mesh(domain, triangle):
       plt.triplot(domain[:,0], domain[:,1], triangle.simplices.copy())
       plt.plot(domain[:,0], domain[:,1], "o")

       # to see the nodes on the graph
       # enumerate shows which node number corresponds to the coordinates
       for i, (x, y) in enumerate(domain):
              plt.text(x, y, f"P{i}", fontsize=9)

       # labeling the triangles
       for k, tri in enumerate(triangle.simplices):

              centroid = domain[tri].mean(axis=0)

              plt.text(centroid[0], centroid[1], f"T{k}", color="red", fontsize=6)

       plt.gca().set_title("Mesh visualisation")
       # gca - get current axes
       # set_aspect("equal") prevents stretching the plot, if it's square it will look like square
       plt.gca().set_aspect("equal")
       plt.show()

# visualise the FEM function reconstructed from the nodal values
def visualise_FEM(eigenvectors, domain, tri, n_nodes, interior_nodes):
       for k in range(eigenvectors.shape[1]):
              
              v_k = eigenvectors[:, k]

              u = np.zeros(n_nodes)
              u[interior_nodes] = v_k

              plt.tripcolor(
                     domain[:, 0],
                     domain[:, 1],
                     tri.simplices,
                     u,
                     shading="gouraud" #flat
              )

              plt.colorbar()
              plt.gca().set_aspect("equal")
              plt.gca().set_title(f"Eigenfunction with {k+1}-th eigenvector")
              
              plt.show()

              if k == 2:
                     break
              



def visualise_convergence(dof, errors):
       fig, ax = plt.subplots(figsize=(10, 8))
       ax.plot(dof, errors, marker="o")
       ax.set_xlabel(r"Degrees of Freedom")
       ax.set_ylabel(r"$E(h) = |\lambda_1 - \lambda_1^h|$")
       ax.set_title("Convergence of the FEM eigenvalues")
       ax.grid(True)
       plt.show()

def visualise_convergence_rate(base, h, errors):
       fig, ax = plt.subplots(figsize=(10, 8))
       ax.plot(np.emath.logn(base, h), np.emath.logn(base, errors), marker="o")
       ax.set_xlabel(f"log_{base:.3f}(h)")
       ax.set_ylabel(f"log_{base:.3f} (E(h))")
       ax.set_title("Convergence rate of the FEM eigenvalues")
       ax.grid(True)
       plt.show()

def visualise_eigenfunctions(discrete_eigf, exact_eigf, interior_nodes):
       
       # reshape eigenvectors into square arrays
       interior_dim = int(np.sqrt(len(interior_nodes)))
       discrete_eigf = discrete_eigf.reshape(interior_dim, interior_dim)
       exact_eigf = exact_eigf.reshape(interior_dim, interior_dim)
       
       fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
       fig.suptitle("Numerical Eigenfunction & Exact Eigenfunction")
       
       ax1.imshow(discrete_eigf)
       ax1.set_title("FEM")

       ax2.imshow(exact_eigf)
       ax2.set_title("Exact")

       plt.show()
       


In [ ]:
# sanity checks
def sanity_check(A_global, M_global):
       print("\nCHECKS FOR MATRIX SYMMETRY AND ROW SUM CHECK\n")
       print(f"Is the global stiffness matrix symmetric? {np.allclose(A_global, A_global.T)}")
       print(f"Is the global mass matrix symmetric? {np.allclose(M_global, M_global.T)}")

       # may have floating point error 
       print(f"Is the row sum of the global stiffness matrix is 0? {np.allclose(A_global.sum(axis=1), 0)}\n")


In [ ]:
# convergence study
def get_local_conv_row(n_points_list, eigval, errors, row_num):
       row_converg = []
       h = 1 / (n_points_list[row_num] - 1)

       p_local, ratio_between_num_points = local_convergence_rate(row_num, n_points_list, errors)
              
       row_converg.append(h)
       row_converg.append(eigval)
       row_converg.append(errors[row_num])
       row_converg.append(p_local)

       return row_converg, ratio_between_num_points

# find local convergence rate
def local_convergence_rate(row_num, n_points_list, errors):

       if row_num == 0:
              
              return 0, 2
              
       # prevents the cases when the error is 0
       e_old = errors[row_num-1]
       e_new = errors[row_num]
       if e_old <= 0 or e_new <= 0:
              return np.nan, np.nan

       
       
       # we need to find the base for the logarithm first
       ratio_between_num_points = n_points_list[row_num] / n_points_list[row_num-1]
              
       # find the ratio between 2 local errors
       ratio_between_errors = errors[row_num-1] / errors[row_num]

       # calculate the local rate of convergence
       p_local = math.log(ratio_between_errors, ratio_between_num_points)

       return p_local, ratio_between_num_points



In [ ]:
# construction of the row for table
def get_row(n_points, interior_nodes, eigval, error):
       row = [] 
       
       row.append(n_points)
       row.append(len(interior_nodes))
       row.append(eigval)
       row.append(error)

       return row

In [ ]:
# compare numerical and exact eigenfunctions
# for the unit square the exact eigenfunction is: 
# u(x,y) = sin(pi * x) * sin(pi * y)
# we need to evaluate it at every node and compare with first eigenvector

def compare_numerical_with_exact(node_coords, u_h):
       
       u_exact = []

       for node_coord in node_coords:
              x = node_coord[0]
              y = node_coord[1]
              
              # calculate the exact eigenfunction
              u_exact.append(np.sin(np.pi * x) * np.sin(np.pi * y))

              
       u_exact = np.array(u_exact)

       # handle the sign
       if np.dot(u_exact, u_h) < 0:
              u_h = -u_h

       # normalise 
       u_exact /= np.linalg.norm(u_exact)
       u_h /= np.linalg.norm(u_h)
       
       return u_exact, u_h

def compare_second_third_eigf_with_exact(node_coords, u_h_2, u_h_3):
       exact_12 = []
       exact_21 = []

       for node_coord in node_coords:
              x = node_coord[0]
              y = node_coord[1]
              
              # calculate the exact eigenfunction
              exact_12.append(np.sin(np.pi * x) * np.sin(2 * np.pi * y))
       
              exact_21.append(np.sin(2 * np.pi * x) * np.sin(np.pi * y))

       exact_12 = np.array(exact_12)
       exact_21 = np.array(exact_21)

       exact_12 /= np.linalg.norm(exact_12)
       exact_21 /= np.linalg.norm(exact_21)
       u_h_2 /= np.linalg.norm(u_h_2)
       u_h_3 /= np.linalg.norm(u_h_3)

       return exact_12, exact_21, u_h_2, u_h_3

In [ ]:
def run_FEM_analysis(n_points, n_points_list, errors, errors_second_third, errors_eigf, index):
       
# create mesh
       domain, triangle = generate_mesh(n_points)
       
       tri_coord_sort = np.sort(triangle.simplices)

       if n_points < 15:
              visualise_mesh(domain, triangle)

# get global matrices
       n_nodes = len(domain)
       A_global, M_global = get_global_matrices(tri_coord_sort, domain, n_nodes)
       # check if the matrices are symmetric and if the row sum is 0
       sanity_check(A_global, M_global)

# Boundary condition
       boundary_nodes, interior_nodes = get_boundary_and_interior_nodes(domain)
       # apply dirichlet BC
       A_reduced, M_reduced = apply_dirichlet(A_global, M_global, interior_nodes)

# finding eigenvalues
       eigvals, eigvecs = eigh(A_reduced, M_reduced)
       # SciPy stores eigvectors as columns

       print(f"The eigenvalues: \n{eigvals[0]}\n")

# eigenvalue error
       error = first_eigval_error(eigvals[0])
       errors.append(error)
       error_23 = second_third_eigval_error(eigvals[1], eigvals[2])
       errors_second_third.append(error_23)

# eigenfunction comparison
       interior_nodes_coords = [domain[interior_node] for interior_node in interior_nodes] 
       exact_eigf, discrete_eigf = compare_numerical_with_exact(interior_nodes_coords, eigvecs[:,0])
       
       # compare second and third with the exact
       exact_12, exact_21, v2, v3 = compare_second_third_eigf_with_exact(interior_nodes_coords, eigvecs[:,1], eigvecs[:, 2])

       # get the error
       error_eigf = np.linalg.norm(exact_eigf - discrete_eigf)
       errors_eigf.append(error_eigf)

# convergence
       h = 1 / (n_points - 1)
       p_eigval, _ = local_convergence_rate(index, n_points_list, errors)
       p_eigf, _ = local_convergence_rate(index, n_points_list, errors_eigf)

# visualise FEM
       visualise_FEM(eigvecs, domain, triangle, n_nodes, interior_nodes)

# visualise exact eigenfunction next to FEM eigenfunction
       visualise_eigenfunctions(discrete_eigf, exact_eigf, interior_nodes)
       

       return {
              "h": h,
              "n_points": n_points,
              "dofs": len(interior_nodes),
              "eigval": eigvals[0],
              "error_eigval": error,
              "error_23": errors_second_third,
              "error_eigf": error_eigf,
              "p_eigval": p_eigval,
              "p_eigf": p_eigf,
              "second_eigvec_second_eigf": abs(np.dot(v2, exact_12)),
              "second_eigvec_third_eigf": abs(np.dot(v2, exact_21)),
              "third_eigvec_second_eigf": abs(np.dot(v3, exact_12)),
              "third_eigvec_third_eigf": abs(np.dot(v3, exact_21))

       }

In [ ]:
# More complex mesh 
#######
##### MAIN #####
########

n_points_list = [4, 6, 12, 24, 48]

errors = []
errors_second_third = []
errors_eigf = []

# TABLES
# error table
error_table = pd.DataFrame(columns=["Points per axis", "Interior DoF", "First eigenvalue", "Error"])
# convergence rate table
conv_table = pd.DataFrame(columns=["h", "lambda_1,h", "error", "rate of convergence"])
# eigenfunction error table
eigf_error_table = pd.DataFrame(columns=["Number of Elements", "Error"])
# eigenfunction convergence table
eigf_conv_table = pd.DataFrame(columns=["h", "Error", "Rate of convergence"])

for index, n_points in enumerate(n_points_list):
       result = run_FEM_analysis(n_points, n_points_list, errors, errors_second_third, errors_eigf, index)

       # put results in the table
       error_table.loc[index + 1] = [result["n_points"], result["dofs"], result["eigval"], result["error_eigval"]]
       conv_table.loc[index + 1] = [result["h"], result["eigval"], result["p_eigval"], result["error_eigval"]]
       eigf_error_table.loc[index + 1] = [result["n_points"], result["error_eigf"]]
       eigf_conv_table.loc[index + 1] = [result["h"], result["error_eigf"], result["p_eigf"]]

       print(f"Second eigenvector dot second eigenfunction {result['second_eigvec_second_eigf']}\n")
       print(f"Second eigenvector dot third eigenfunction {result['second_eigvec_third_eigf']}\n")
       print(f"Third eigenvector dot second eigenfunction {result['third_eigvec_second_eigf']}\n")
       print(f"Third eigenvector dot third eigenfunction {result['third_eigvec_third_eigf']}\n")


# Error table
print(f"\nERROR DEPENDENCY ON THE NUMBER POINTS PER AXIS\n{error_table}")
# Convergence rate
print(f"\nCONVERGENCE RATE TABLE\n {conv_table}")
# Error between the exact and numerical eigenfunctions
print(f"\nERROR BETWEEN EXACT AND NUMERICAL EIGENFUNCTIONS\n{eigf_error_table}")
# convergence rate of the eigenfucntions
print(f"\nEIGENFUNCTION CONVERGENCE RATE TABLE\n{eigf_conv_table}\n")


# Plots
dofs = error_table["Interior DoF"].tolist()
hs = conv_table["h"].tolist()

ratio_between_num_points = [n_points_list[i] / n_points_list[i-1] for i in range(1, len(n_points_list))] 
base = np.mean(ratio_between_num_points)


visualise_convergence(dofs, errors)
       
# plotting convergence rate of eigenvalues
visualise_convergence_rate(base, hs, errors)

# plotting convergence rate of eigenvectors
visualise_convergence_rate(base, hs, errors_eigf)

p_global = np.nanmean(eigf_conv_table["Rate of convergence"].tolist())
print(f"\nThe convergence rate is {p_global}")


**RESULTS AND OBSEVATIONS**
- $\lambda_1$ decreases with the better mesh
- the error decreases because the finite-dimentional subspace grows and can represent smoother functions
- the eigenfunction has a shape of a smooth hill
- Writing u(x,y) = X(x)Y(y) splits the Laplacian eigenvalue problem into two independent 1D problems, each giving sine solutions. The lowest-frequency solution in each direction is sin(πx) and sin(πy), so the first eigenfunction is exactly their product. FEM solution is a piecewise linear approximation to this — visually indistinguishable on fine meshes
- if v is an eigenvector, then -v also will satisfy the eigenvalue problem. the correction is made by checking the dot product: if the dot product is negative, the vectors point in opposite directions, so the sign of u_h is flipped before computing the error. This ensures the measurement of the true approximation error.